In [1]:
import os
import pandas as pd
from sklearn.metrics import confusion_matrix
from sklearn.metrics import matthews_corrcoef
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_curve, roc_auc_score, classification_report, auc
from sklearn.metrics import precision_recall_curve

In [59]:
pwd

'/homes/t326h379'

# I want to check what sites of B4DU15 are ubiquitinated

# Please enter the protein that you want to analyze

In [2]:
name_of_the_protein = input("Enter the name of the protein: ")

Enter the name of the protein: B4DU15


# Load the Protein File

In [31]:
import requests

url = "https://rest.uniprot.org/uniprotkb/"+str(name_of_the_protein)+".fasta"

# Send GET request
response = requests.get(url)

# Check if the request was successful
if response.status_code == 200:
    # Save the content to a local file
    
    fasta_file_name = name_of_the_protein+".fasta"
    
    with open(fasta_file_name, "w") as file:
        file.write(response.text)
    print("FASTA file downloaded successfully.")
    
else:
    print(f"Failed to download file. Status code: {response.status_code}")

FASTA file downloaded successfully.


In [32]:
all_amino_acid = []
with open(fasta_file_name) as Houston:
    for line in Houston:
        x = line.strip("\n")
        all_amino_acid.append(x)

all_amino_acid = all_amino_acid[1:]
Ammino_Acid_in_Protein = ''.join(all_amino_acid)

In [33]:
Ammino_Acid_in_Protein

'MSVACVLKRKAVLWQDSFSPHLKHHPQEPANPNMPVVLTSGTGSQAQPQPAANQALAAGTHSSPVPGSIGVAGRSQDDAMVDYFFQRQHGEQLGGGGSGGGGYNNSKHRWPTGDNIHAEHQVRSMDELNHDFQALALEGRAMGEGPRDADSDENDKGEKKNKGTFDGDKLGDLKEEGDVMDKTNGLPVQNGIDADVKDFSRTPGNCQNSANEVDLLGPNQNGSEGLAQLTSTNGAKPVEDFSNMESQSVPLDPMEHVGMEPLQFDYSGTQVPVDSAAATVGLFDYNSQQQLFQRPNALAVQQLTAAQQQQYALAAAHQPHIGLAPAAFVPNPYIISAAPPGTDPYTAGLAAAATLGPAVVPHQYYGVTPWGVYPASLFQQQAAAAAAATNSANQQTTPQAQQGQQQVLRGGASQRPLTPNQNQQGQQTDPLVAAAAVNSALAFGQGLAAGMPAVAAAAASANGAAGGLAGTTNGPFRPLGTQQPQPQPQQQPNNNLASSSFYGNNSLNSNSQSSSLFSQGSAQPANTSLGFGSSSFLGATLGSALGGFGTAVANSNTGSGSRRDSLTGSSDLYKRTSSSLTPIGHSFYNGLSFSSSPGPVGMPLPSQGPGHSQTPPPSLSSHGSSSSLNLGGLTNGSGRYISAAPGAEAKYRSASSASSLFSPSSTLFSSSRLRYGMSDVMPSGRSRLLEDFRNNRYPNLQLREIAGHIMEFSQDQHGSRFIQLKLERATPAERQLVFNEILQAAYQLMVDVFGNYVIQKFFEFGSLEQKLALAERIRGHVLSLALQMYGCRVIQKALEFIPSDQQVINEMVRELDGHVLKCVKDQNGNHVVQKCIECVQPQSLQFIIDAFKGQVFALSTHPYGCRVIQRILEHCLPDQTLPILEELHQHTEQLVQDQYGNYVIQHVLEHGRPEDKSKIVAEIRGNVLVLSQHKFASNVVEKCVTHASRTERAVLIDEVCTMNDGPHSALYTMMKDQYANYVVQKMIDVAEPGQRKIVM

# Find the postion of lysine in protein

In [34]:
positions = [i for i, aa in enumerate(Ammino_Acid_in_Protein) if aa == 'K']
print(positions)

[7, 9, 22, 106, 155, 158, 159, 161, 168, 173, 181, 196, 235, 573, 649, 724, 759, 769, 795, 820, 823, 833, 851, 915, 917, 933, 941, 974, 984, 995, 1000, 1010, 1015, 1020, 1023, 1027]


# Delete the file if it exists

In [35]:
import os

file_path = 'Test_Sequences_81_window_Training.fasta'
if os.path.exists(file_path):
    os.remove(file_path)
    print(f"{file_path} deleted.")
else:
    print("File not found.")

Test_Sequences_81_window_Training.fasta deleted.


# Window Extraction

In [36]:
import re
from Bio import SeqIO

window_size = 81

def open_file_extract_window(pid,position):
    position = int(position)
    fasta_file = pid+".fasta"

    with open("Test_Sequences_81_window_Training.fasta","a+") as fp:
        for seq_record in SeqIO.parse(fasta_file,"fasta"):
            placeholder = str(seq_record.id).split("|")[1]  
            seq = str(seq_record.seq)

            if placeholder == pid:
                position = int(position)

                if seq[position] == "K":

                    half_window = window_size // 2

                    C_terminal_calculation = len(seq) - position
                   
                   
                    if len(seq) > 41 and len(seq) < 81:
                        check_length_of_t_terminal = seq[position+1:]
                        check_length_of_N_terminal = seq[:position]

                        
                        if len(check_length_of_t_terminal) < 41 and len(check_length_of_N_terminal) < 41:
                            half_window_size = 40
                            n_terminal_dummy_to_be_filled = half_window_size-len(check_length_of_N_terminal)
                            fillerr=[]            
                            for x in range(half_window_size):
                                dummy="-"
                                fillerr.append(dummy)
                            select = fillerr[0:int(n_terminal_dummy_to_be_filled)]
                            nTer_adjust = ''.join(select)
                            N_Terminal_needed_sequence = nTer_adjust+check_length_of_N_terminal
                            
                            t_terminal_dummy_to_be_filled = half_window_size-len(check_length_of_t_terminal)
                            t_fill = []
                            for x in range(half_window_size):
                                dummy="-"
                                t_fill.append(dummy)
                            t_select = t_fill[0:int(t_terminal_dummy_to_be_filled)]
                            tTer_adjust = ''.join(t_select)
                            TT_Terminal_needed_sequence = check_length_of_t_terminal+tTer_adjust
                            
                            final_window = N_Terminal_needed_sequence + seq[position] + TT_Terminal_needed_sequence
                            fp.write(">")
                            fp.write(pid)
                            fp.write("|")
                            fp.write(str(position))
                            fp.write("\n")
                            fp.write(final_window)
                            fp.write("\n")                           

                        else:
                            if position < 41:
                                n_terminal_sequence = seq[:position+1]
                                p_terminal_sequence = seq[position+1:41+position]
                                needed_sequence = n_terminal_sequence+p_terminal_sequence
                                dummy_to_be_filled = window_size - len(needed_sequence)

                                fill=[]            
                                for x in range(window_size):
                                    dummy="-"
                                    fill.append(dummy)
                                select = fill[0:int(dummy_to_be_filled)]
                                nTer_adjust = ''.join(select)
                                motif = nTer_adjust +needed_sequence
                                fp.write(">")
                                fp.write(pid)
                                fp.write("|")
                                fp.write(str(position))
                                fp.write("\n")
                                fp.write(motif)
                                fp.write("\n")  

                            if position > 41 or position == 41:
                                index_of_half_window_sequence = position-40
                                n_terminal = seq[index_of_half_window_sequence:position]
                                needed_sequence_we_need = n_terminal+seq[position:]

                                dummy_to_be_filled = window_size - len(needed_sequence_we_need)

                                fill=[]            
                                for x in range(window_size):
                                    dummy="-"
                                    fill.append(dummy)
                                select = fill[0:int(dummy_to_be_filled)]
                                tTer_adjust = ''.join(select)
                                total_length = needed_sequence_we_need+tTer_adjust
                                fp.write(">")
                                fp.write(pid)
                                fp.write("|")
                                fp.write(str(position))
                                fp.write("\n")
                                fp.write(total_length)  
                                fp.write("\n")  


                    if len(seq) > 82 or len(seq) == 82:
                        if position < half_window:
                            # N_terminal_pad
                            dummy_to_be_filled = half_window - position
                            fill=[]            
                            for x in range(window_size):
                                dummy="-"
                                fill.append(dummy)
                            select = fill[0:int(dummy_to_be_filled)]
                            N_terminal_truncated_motif = seq[0:position+half_window+1]
                            nTer_adjust = ''.join(select)
                            motif = nTer_adjust+N_terminal_truncated_motif
                            fp.write(">")
                            fp.write(pid)
                            fp.write("|")
                            fp.write(str(position))
                            fp.write("\n")
                            fp.write(motif)
                            fp.write("\n")

                        if C_terminal_calculation < half_window:
                            # C_terminal_pad
                            C_terminal_motif = seq[position-40:]
                            dummy_to_be_filled = window_size - len(C_terminal_motif)

                            fill=[]            
                            for x in range(window_size):
                                dummy="-"
                                fill.append(dummy)
                            select = fill[0:int(dummy_to_be_filled)]
                            cTer_adjust = ''.join(select)
                            motif = C_terminal_motif+cTer_adjust
                            fp.write(">")
                            fp.write(pid)
                            fp.write("|")
                            fp.write(str(position))
                            fp.write("\n")
                            fp.write(motif)
                            fp.write("\n")


                        if len(seq[int(position)-half_window:int(position)+half_window+1]) == window_size:
                            motif = seq[int(position)-40:int(position)+41]
                            fp.write(">")
                            fp.write(pid)
                            fp.write("|")
                            fp.write(str(position))
                            fp.write("\n")
                            fp.write(motif)
                            fp.write("\n")

In [37]:
for value in positions:    
    open_file_extract_window(name_of_the_protein,value)

In [38]:
Peptides_to_be_Tested = []
with open("Test_Sequences_81_window_Training.fasta") as User:
    for line in User:
        if line.startswith(">"):
            pass
        else:
            Peptides_to_be_Tested.append(line[30:-31])

In [39]:
Peptides_to_be_Tested

['---MSVACVLKRKAVLWQDSF',
 '-MSVACVLKRKAVLWQDSFSP',
 'LWQDSFSPHLKHHPQEPANPN',
 'GSGGGGYNNSKHRWPTGDNIH',
 'PRDADSDENDKGEKKNKGTFD',
 'ADSDENDKGEKKNKGTFDGDK',
 'DSDENDKGEKKNKGTFDGDKL',
 'DENDKGEKKNKGTFDGDKLGD',
 'KKNKGTFDGDKLGDLKEEGDV',
 'TFDGDKLGDLKEEGDVMDKTN',
 'DLKEEGDVMDKTNGLPVQNGI',
 'PVQNGIDADVKDFSRTPGNCQ',
 'LAQLTSTNGAKPVEDFSNMES',
 'DSLTGSSDLYKRTSSSLTPIG',
 'YISAAPGAEAKYRSASSASSL',
 'DQHGSRFIQLKLERATPAERQ',
 'VDVFGNYVIQKFFEFGSLEQK',
 'KFFEFGSLEQKLALAERIRGH',
 'LQMYGCRVIQKALEFIPSDQQ',
 'MVRELDGHVLKCVKDQNGNHV',
 'ELDGHVLKCVKDQNGNHVVQK',
 'KDQNGNHVVQKCIECVQPQSL',
 'QSLQFIIDAFKGQVFALSTHP',
 'HVLEHGRPEDKSKIVAEIRGN',
 'LEHGRPEDKSKIVAEIRGNVL',
 'RGNVLVLSQHKFASNVVEKCV',
 'QHKFASNVVEKCVTHASRTER',
 'GPHSALYTMMKDQYANYVVQK',
 'KDQYANYVVQKMIDVAEPGQR',
 'MIDVAEPGQRKIVMHKIRPHI',
 'EPGQRKIVMHKIRPHIATLRK',
 'KIRPHIATLRKYTYGKHILAK',
 'IATLRKYTYGKHILAKLEKYY',
 'KYTYGKHILAKLEKYYMKNGV',
 'YGKHILAKLEKYYMKNGVDLG',
 'ILAKLEKYYMKNGVDLGPICG']

In [40]:
Final_Test_Peptide = []

for i in range(len(Peptides_to_be_Tested)):
    value = Peptides_to_be_Tested[i]
    if "B" in value or "J" in value or "O" in value or "U" in value or "Z" in value:
        pass   

    else:
        Final_Test_Peptide.append(value)

In [41]:
Final_Test_Peptide

['---MSVACVLKRKAVLWQDSF',
 '-MSVACVLKRKAVLWQDSFSP',
 'LWQDSFSPHLKHHPQEPANPN',
 'GSGGGGYNNSKHRWPTGDNIH',
 'PRDADSDENDKGEKKNKGTFD',
 'ADSDENDKGEKKNKGTFDGDK',
 'DSDENDKGEKKNKGTFDGDKL',
 'DENDKGEKKNKGTFDGDKLGD',
 'KKNKGTFDGDKLGDLKEEGDV',
 'TFDGDKLGDLKEEGDVMDKTN',
 'DLKEEGDVMDKTNGLPVQNGI',
 'PVQNGIDADVKDFSRTPGNCQ',
 'LAQLTSTNGAKPVEDFSNMES',
 'DSLTGSSDLYKRTSSSLTPIG',
 'YISAAPGAEAKYRSASSASSL',
 'DQHGSRFIQLKLERATPAERQ',
 'VDVFGNYVIQKFFEFGSLEQK',
 'KFFEFGSLEQKLALAERIRGH',
 'LQMYGCRVIQKALEFIPSDQQ',
 'MVRELDGHVLKCVKDQNGNHV',
 'ELDGHVLKCVKDQNGNHVVQK',
 'KDQNGNHVVQKCIECVQPQSL',
 'QSLQFIIDAFKGQVFALSTHP',
 'HVLEHGRPEDKSKIVAEIRGN',
 'LEHGRPEDKSKIVAEIRGNVL',
 'RGNVLVLSQHKFASNVVEKCV',
 'QHKFASNVVEKCVTHASRTER',
 'GPHSALYTMMKDQYANYVVQK',
 'KDQYANYVVQKMIDVAEPGQR',
 'MIDVAEPGQRKIVMHKIRPHI',
 'EPGQRKIVMHKIRPHIATLRK',
 'KIRPHIATLRKYTYGKHILAK',
 'IATLRKYTYGKHILAKLEKYY',
 'KYTYGKHILAKLEKYYMKNGV',
 'YGKHILAKLEKYYMKNGVDLG',
 'ILAKLEKYYMKNGVDLGPICG']

In [42]:
len(Final_Test_Peptide)

36

# AAindex Encoding

In [43]:
aminoacids='ARNDCQEGHILKMFPSTWYV-'
import numpy as np
aaindex=pd.read_table('aaindex31.txt',sep='\s+',header=None)
aaindex=aaindex.subtract(aaindex.min(axis=1),axis=0).divide((aaindex.max(axis=1)-aaindex.min(axis=1)),axis=0)
aa=[x for x in 'ARNDCQEGHILKMFPSTWYV']
aaindex=aaindex.to_numpy().T
index={x:y for x,y in zip(aa,aaindex.tolist())}
index['-']=np.zeros(31).tolist()
index['X']=np.zeros(31).tolist()

def index_encode(Sequence_one_hot_encoding):
    encoding=[]
    for i in range(len(Sequence_one_hot_encoding)):
        s=Sequence_one_hot_encoding[i]
        encoding.append([index[x] for x in (s)])
    encoding=np.array(encoding)
    return encoding

In [44]:
# Independent Test AAindex Encoding
AA_Index_X_Test = index_encode(list(Final_Test_Peptide))

In [45]:
AA_Index_X_Test.shape

(36, 21, 31)

# One Hot Encoding

In [46]:
def binary_encode(Sequence_one_hot_encoding): 
    aa2v={x:y for x,y in zip(aminoacids,np.eye(21,21).tolist())}
    aa2v['X']=np.zeros(21)
    encoding=[]
    for i in range(len(Sequence_one_hot_encoding)):
        s=Sequence_one_hot_encoding[i]
        encoding.append([aa2v[x] for x in s])
    encoding=np.array(encoding)
    return encoding

binaryCoding_X_Test = binary_encode(list(Final_Test_Peptide))

In [47]:
binaryCoding_X_Test.shape

(36, 21, 21)

# Embedding Encoding

In [48]:
updated_aminoacids = aminoacids
char_to_int = dict((c, i) for i, c in enumerate(updated_aminoacids))

test_x_21 = []

def inner1(data):
    for char in data:
        if char not in updated_aminoacids:
            print(data)
            return

    integer_encoded = [char_to_int[char] for char in data]
    test_x_21.append(integer_encoded)

for value in list(Final_Test_Peptide):
    inner1(value)

EmbeddingCoding_X_Test = np.array(test_x_21)

In [49]:
EmbeddingCoding_X_Test.shape

(36, 21)

# Load the trained model

In [50]:
from tensorflow import keras

# Load the entire model (architecture + weights + optimizer state)
model = keras.models.load_model('Shrestha_et_al_AAindex_one_hot_and_keras_embedding42.h5')

In [51]:
pwd

'/homes/t326h379'

In [52]:
model.summary()

Model: "model_10"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_33 (InputLayer)          [(None, 21)]         0           []                               
                                                                                                  
 embedding_10 (Embedding)       (None, 21, 21)       483         ['input_33[0][0]']               
                                                                                                  
 conv1d_31 (Conv1D)             (None, 19, 256)      16384       ['embedding_10[0][0]']           
                                                                                                  
 max_pooling1d_31 (MaxPooling1D  (None, 9, 256)      0           ['conv1d_31[0][0]']              
 )                                                                                         

In [53]:
X_independent = [AA_Index_X_Test, binaryCoding_X_Test, EmbeddingCoding_X_Test]
Y_pred = model.predict(X_independent)
Y_pred = (Y_pred > 0.5)
y_pred = [np.argmax(y, axis=None, out=None) for y in Y_pred]
y_pred = np.array(y_pred)

2/2 [==============================] - 0s 3ms/step


In [54]:
len(y_pred)

36

In [55]:
len(positions)

36

In [56]:
repeat_names = []

for i in range(len(positions)):

    repeat_names.append(name_of_the_protein)

In [57]:
df = pd.DataFrame({
    'Name of protein': repeat_names,
    'Positions of K': positions,
    'Prediction': y_pred
})

In [58]:
df

,Name of protein,Positions of K,Prediction
0,B4DU15,7,0
1,B4DU15,9,1
2,B4DU15,22,0
3,B4DU15,106,1
4,B4DU15,155,0
5,B4DU15,158,0
6,B4DU15,159,0
7,B4DU15,161,1
8,B4DU15,168,1
9,B4DU15,173,1


# Thank You